# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [5]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent



Create a simple Nutrition Assistant Agent

In [7]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """
)

Let's execute the Agent:

In [8]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Overall, bananas are healthy in moderation.
    
    Key nutrients per medium banana:
    - Potassium: supports blood pressure and heart function
    - Vitamin B6: helps metabolism and brain function
    - Vitamin C and dietary fiber: antioxidant and digestive benefits
    - Low in fat and protein; natural sugars are present
    
    Considerations:
    - Moderate glycemic impact; ripeness increases sugar (ripe bananas sweeter, softer)
    - diabetics should watch portions and pair with protein/fat
    - not a complete protein; variety is important
    
    Bottom line: a convenient, nutrient-dense snack or part of a balanced meal. Pair with protein or fats for fullness.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [9]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Pretty healthy as part of a balanced diet.

Key points:
- Nutrients: good source of potassium, vitamin B6, vitamin C, and dietary fiber.
- Energy: about 100 calories per medium banana; natural sugars, so portion matters if you’re watching sugar intake.
- Unripe vs ripe: unripe (green) bananas have more resistant starch (helps fullness); ripe bananas are sweeter and easier to digest.
- Mix: pair with protein or fat (peanut butter, yogurt) to improve satiety and blood sugar response.
- Cautions: portion size for diabetes or strict sugar goals; rare latex-fruit allergy can cause cross-reaction.

Bottom line: 1–2 bananas a day fit most healthy diets.

_Good Job!_